# Comparativo global: MASTER, TFB, RandomTopJ e benchmarks

Este notebook consolida os JSONs de resultados e, quando houver `sinais.csv` na mesma pasta, incorpora as AUCs direcionais.

**Ponto importante de interpretação:** TFB e MASTER podem ter `pred_len` com significado diferente. Por isso, a comparação principal deve ser feita por `janela_trading`/`horizonte_comparavel` (`k` de rebalanceamento), não apenas por `pred_len`.


In [1]:
from pathlib import Path
import sys
import re
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'utils':
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.comparativo_metricas_global import comparar_global, METRICAS_INTERESSE

OUT = ROOT / 'simulacoes' / 'comparativo_global_master_tfb'
OUT


PosixPath('/sonic_home/igor.viveiros/paralelo/simulacoes/comparativo_global_master_tfb')

In [3]:
dfs = comparar_global(
    base_dir=ROOT,
    output_dir='simulacoes/comparativo_global_master_tfb',
    top_n=30,
)

metricas = dfs['metricas']
resumo = dfs['resumo']
top_configs = dfs['top_configs']
ranking = dfs['ranking']

neg_master = pd.read_csv(
    "/sonic_home/igor.viveiros/paralelo/simulacoes/master_tfb_experimento/acerto_negativos_master.csv"
)

metricas["output_dir"] = metricas["json_path"].apply(lambda p: str(Path(p).parent))

metricas = metricas.drop(
    columns=["taxa_acerto_negativos", "mean_precision_negative"],
    errors="ignore"
)

metricas = metricas.merge(
    neg_master[[
        "output_dir",
        "taxa_acerto_negativos",
        "mean_precision_negative",
        "n_pred_negativos",
        "n_acertos_negativos",
        "n_janelas_com_negativos",
    ]],
    on="output_dir",
    how="left"
)

def simplificar_dataset_master(nome):
    nome = str(nome)

    # ordem importa: log_return antes de return
    if re.search(r"(__|_|-)log_returns?$", nome):
        return "log_return"
    if re.search(r"(__|_|-)returns?$", nome):
        return "return"
    if re.search(r"(__|_|-)prices?$", nome):
        return "prices"

    return nome

def corrigir_datasets_master(df):
    if df is None or df.empty or "dataset" not in df.columns:
        return df

    df = df.copy()

    if "grupo" in df.columns:
        mask = df["grupo"].eq("MASTER")
        df.loc[mask, "dataset"] = df.loc[mask, "dataset"].apply(simplificar_dataset_master)
    else:
        df["dataset"] = df["dataset"].apply(simplificar_dataset_master)

    return df

metricas = corrigir_datasets_master(metricas)
resumo = corrigir_datasets_master(resumo)
top_configs = corrigir_datasets_master(top_configs)
ranking = corrigir_datasets_master(ranking)
print(f'Linhas carregadas: {len(metricas)}')
print(f'Arquivos salvos em: {OUT}')
metricas.head()


/sonic_home/igor.viveiros/paralelo/utils/auc_direcional.py:87: RuntimeWarning: Mean of empty slice
  "auc_alta_media_janela": float(np.nanmean(auc_alta_janelas)) if auc_alta_janelas else np.nan,
/sonic_home/igor.viveiros/paralelo/utils/auc_direcional.py:88: RuntimeWarning: Mean of empty slice
  "auc_queda_media_janela": float(np.nanmean(auc_queda_janelas)) if auc_queda_janelas else np.nan,
/sonic_home/igor.viveiros/paralelo/utils/auc_direcional.py:87: RuntimeWarning: Mean of empty slice
  "auc_alta_media_janela": float(np.nanmean(auc_alta_janelas)) if auc_alta_janelas else np.nan,
/sonic_home/igor.viveiros/paralelo/utils/auc_direcional.py:88: RuntimeWarning: Mean of empty slice
  "auc_queda_media_janela": float(np.nanmean(auc_queda_janelas)) if auc_queda_janelas else np.nan,
/sonic_home/igor.viveiros/paralelo/utils/auc_direcional.py:87: RuntimeWarning: Mean of empty slice
  "auc_alta_media_janela": float(np.nanmean(auc_alta_janelas)) if auc_alta_janelas else np.nan,
/sonic_home/igor.vi

Linhas carregadas: 1062
Arquivos salvos em: /sonic_home/igor.viveiros/paralelo/simulacoes/comparativo_global_master_tfb


,grupo,dataset,modelo,lookback,pred_len,janela_trading,horizonte_comparavel,max_assets,model_output,returns_mode,...,max_drawdown,icir,mean_n_assets,n_periods,output_dir,taxa_acerto_negativos,mean_precision_negative,n_pred_negativos,n_acertos_negativos,n_janelas_com_negativos
0,Benchmark,meu_multi_lb_predlen_todos_ativos,k_1,32,1,1,1,9,NaN,NaN,...,-0.15518,NaN,66.0,569.0,/sonic_home/igor.viveiros/paralelo/simulacoes/...,NaN,NaN,NaN,NaN,NaN
1,Benchmark,meu_multi_lb_predlen_todos_ativos,k_1,32,1,1,1,9,NaN,NaN,...,-0.15518,NaN,66.0,560.0,/sonic_home/igor.viveiros/paralelo/simulacoes/...,NaN,NaN,NaN,NaN,NaN
2,Benchmark,meu_multi_lb_predlen_todos_ativos,k_1,32,1,1,1,9,NaN,NaN,...,-0.15518,NaN,66.0,555.0,/sonic_home/igor.viveiros/paralelo/simulacoes/...,NaN,NaN,NaN,NaN,NaN
3,Benchmark,meu_multi_lb_predlen_todos_ativos,k_1,32,1,1,1,9,NaN,NaN,...,-0.15518,NaN,66.0,550.0,/sonic_home/igor.viveiros/paralelo/simulacoes/...,NaN,NaN,NaN,NaN,NaN
4,Benchmark,meu_multi_lb_predlen_todos_ativos,k_1,32,1,1,1,9,NaN,NaN,...,-0.15518,NaN,66.0,546.0,/sonic_home/igor.viveiros/paralelo/simulacoes/...,NaN,NaN,NaN,NaN,NaN


In [4]:
# Cobertura por grupo
if not metricas.empty:
    display(metricas.groupby('grupo').size().rename('n_resultados').reset_index())
    display(metricas.groupby(['grupo', 'modelo']).size().rename('n_resultados').reset_index().sort_values(['grupo', 'n_resultados'], ascending=[True, False]).head(50))


,grupo,n_resultados
0,Benchmark,252
1,MASTER,108
2,TFB,702


,grupo,modelo,n_resultados
0,Benchmark,k_1,72
5,Benchmark,k_5,60
1,Benchmark,k_10,48
2,Benchmark,k_15,36
3,Benchmark,k_20,24
4,Benchmark,k_24,12
6,MASTER,MASTER,108
7,TFB,DUET,144
9,TFB,FEDformer,144
11,TFB,Nonstationary_Transformer,144


In [5]:
# Tabela principal das métricas de interesse
cols = ['grupo', 'dataset', 'modelo', 'lookback', 'pred_len', 'janela_trading', 'max_assets', *METRICAS_INTERESSE, 'json_path']
tabela = metricas[[c for c in cols if c in metricas.columns]].copy()
tabela.sort_values(['janela_trading', 'mean_spearman_ic'], ascending=[True, False], na_position='last').head(50)


,grupo,dataset,modelo,lookback,pred_len,janela_trading,max_assets,mean_spearman_ic,mean_precision_positive,mean_precision_negative,auc_alta,auc_queda,auc_alta_media_janela,auc_queda_media_janela,json_path
266,MASTER,prices,MASTER,32,1,1,5,0.155638,0.690656,NaN,0.607165,0.600261,0.583364,0.571891,/sonic_home/igor.viveiros/paralelo/simulacoes/...
284,MASTER,prices,MASTER,32,1,1,5,0.129487,0.625561,NaN,0.578656,0.581502,0.565620,0.567855,/sonic_home/igor.viveiros/paralelo/simulacoes/...
254,MASTER,prices,MASTER,104,1,1,5,0.119762,0.525798,NaN,0.570166,0.578075,0.553851,0.563766,/sonic_home/igor.viveiros/paralelo/simulacoes/...
793,TFB,prices,Nonstationary_Transformer,104,1,1,9,0.107185,0.540524,NaN,0.543317,0.543401,0.555463,0.554994,/sonic_home/igor.viveiros/paralelo/simulacoes/...
746,TFB,prices,FEDformer,104,1,1,9,0.104607,0.540440,NaN,0.542058,0.542533,0.555548,0.555776,/sonic_home/igor.viveiros/paralelo/simulacoes/...
742,TFB,prices,FEDformer,104,1,1,9,0.103465,0.542300,NaN,0.541209,0.542054,0.554797,0.555273,/sonic_home/igor.viveiros/paralelo/simulacoes/...
840,TFB,prices,TimesNet,104,1,1,9,0.103083,0.546941,NaN,0.532204,0.531505,0.554163,0.553591,/sonic_home/igor.viveiros/paralelo/simulacoes/...
791,TFB,prices,Nonstationary_Transformer,104,1,1,9,0.101637,0.541582,NaN,0.530920,0.530835,0.551421,0.551064,/sonic_home/igor.viveiros/paralelo/simulacoes/...
744,TFB,prices,FEDformer,104,1,1,9,0.101417,0.533173,NaN,0.539774,0.540067,0.553003,0.552464,/sonic_home/igor.viveiros/paralelo/simulacoes/...
842,TFB,prices,TimesNet,104,1,1,9,0.100662,0.543902,NaN,0.534391,0.533837,0.548490,0.547586,/sonic_home/igor.viveiros/paralelo/simulacoes/...


In [6]:
# Melhores configurações por métrica
top_configs.head(80)


,metrica,valor,grupo,dataset,modelo,lookback,pred_len,janela_trading,max_assets,run,json_path
0,mean_spearman_ic,0.884293,MASTER,prices,MASTER,32,10,10,5,b3_daily_tfb__master_ohlcv_market__lb32__h10__...,/sonic_home/igor.viveiros/paralelo/simulacoes/...
1,mean_spearman_ic,0.782661,MASTER,prices,MASTER,104,10,10,5,b3_daily_tfb__master_ohlcv_market__lb104__h10_...,/sonic_home/igor.viveiros/paralelo/simulacoes/...
2,mean_spearman_ic,0.742626,MASTER,prices,MASTER,32,10,10,5,b3_daily_tfb__master_full_alpha158_ohlcv_marke...,/sonic_home/igor.viveiros/paralelo/simulacoes/...
3,mean_spearman_ic,0.694789,MASTER,prices,MASTER,32,20,20,5,b3_daily_tfb__master_ohlcv_market__lb32__h20__...,/sonic_home/igor.viveiros/paralelo/simulacoes/...
4,mean_spearman_ic,0.640808,MASTER,prices,MASTER,246,20,20,5,b3_daily_tfb__master_ohlcv_market__lb246__h20_...,/sonic_home/igor.viveiros/paralelo/simulacoes/...
...,...,...,...,...,...,...,...,...,...,...,...
75,auc_alta,0.677410,TFB,prices,TimesNet,104,24,24,9,k_24,/sonic_home/igor.viveiros/paralelo/simulacoes/...
76,auc_alta,0.675077,TFB,retornos_simples,TimesNet,104,24,24,9,k_24,/sonic_home/igor.viveiros/paralelo/simulacoes/...
77,auc_alta,0.673194,TFB,prices,FEDformer,104,20,20,9,k_20,/sonic_home/igor.viveiros/paralelo/simulacoes/...
78,auc_alta,0.671390,TFB,prices,Nonstationary_Transformer,32,20,20,9,k_20,/sonic_home/igor.viveiros/paralelo/simulacoes/...


In [7]:
# Ranking médio simples nas métricas de interesse: quanto menor, melhor.
ranking_cols = ['grupo', 'dataset', 'modelo', 'lookback', 'pred_len', 'janela_trading', 'max_assets', 'rank_medio_metricas_interesse', *METRICAS_INTERESSE, 'json_path']
ranking[[c for c in ranking_cols if c in ranking.columns]].head(50)


,grupo,dataset,modelo,lookback,pred_len,janela_trading,max_assets,rank_medio_metricas_interesse,mean_spearman_ic,mean_precision_positive,mean_precision_negative,auc_alta,auc_queda,auc_alta_media_janela,auc_queda_media_janela,json_path
804,TFB,prices,Nonstationary_Transformer,104,20,20,9,6.833333,0.494539,0.797619,NaN,0.695836,0.695768,0.727603,0.727284,/sonic_home/igor.viveiros/paralelo/simulacoes/...
805,TFB,prices,Nonstationary_Transformer,104,24,24,9,7.666667,0.459230,0.729469,NaN,0.709597,0.710395,0.741231,0.741911,/sonic_home/igor.viveiros/paralelo/simulacoes/...
789,TFB,prices,Nonstationary_Transformer,32,24,24,9,10.500000,0.430779,0.719807,NaN,0.698378,0.699109,0.734881,0.735459,/sonic_home/igor.viveiros/paralelo/simulacoes/...
741,TFB,prices,FEDformer,32,24,24,9,10.666667,0.422740,0.719807,NaN,0.704963,0.705770,0.731434,0.732106,/sonic_home/igor.viveiros/paralelo/simulacoes/...
740,TFB,prices,FEDformer,32,20,20,9,11.000000,0.455820,0.765873,NaN,0.686084,0.686496,0.713053,0.713346,/sonic_home/igor.viveiros/paralelo/simulacoes/...
788,TFB,prices,Nonstationary_Transformer,32,20,20,9,16.833333,0.446626,0.734127,NaN,0.671390,0.671834,0.707292,0.707669,/sonic_home/igor.viveiros/paralelo/simulacoes/...
693,TFB,prices,DUET,32,24,24,9,17.166667,0.384581,0.705314,NaN,0.687693,0.688372,0.712677,0.713302,/sonic_home/igor.viveiros/paralelo/simulacoes/...
756,TFB,prices,FEDformer,104,20,20,9,17.166667,0.440102,0.734127,NaN,0.673194,0.673524,0.705090,0.705352,/sonic_home/igor.viveiros/paralelo/simulacoes/...
853,TFB,prices,TimesNet,104,24,24,9,17.333333,0.393942,0.719807,NaN,0.677410,0.678186,0.723202,0.724047,/sonic_home/igor.viveiros/paralelo/simulacoes/...
709,TFB,prices,DUET,104,24,24,9,18.000000,0.384557,0.714976,NaN,0.685375,0.686027,0.711847,0.712489,/sonic_home/igor.viveiros/paralelo/simulacoes/...


In [8]:
# Resumo agregado por grupo/modelo/janela de trading
resumo.sort_values(['janela_trading', 'mean_spearman_ic_mean'], ascending=[True, False], na_position='last').head(80)


,grupo,dataset,modelo,janela_trading,mean_spearman_ic_mean,mean_spearman_ic_median,mean_spearman_ic_std,mean_spearman_ic_count,mean_precision_positive_mean,mean_precision_positive_median,...,auc_queda_std,auc_queda_count,auc_alta_media_janela_mean,auc_alta_media_janela_median,auc_alta_media_janela_std,auc_alta_media_janela_count,auc_queda_media_janela_mean,auc_queda_media_janela_median,auc_queda_media_janela_std,auc_queda_media_janela_count
26,MASTER,prices,MASTER,1,0.155638,0.155638,NaN,1,0.690656,0.690656,...,NaN,1,0.583364,0.583364,NaN,1,0.571891,0.571891,NaN,1
44,MASTER,prices,MASTER,1,0.129487,0.129487,NaN,1,0.625561,0.625561,...,NaN,1,0.565620,0.565620,NaN,1,0.567855,0.567855,NaN,1
14,MASTER,prices,MASTER,1,0.119762,0.119762,NaN,1,0.525798,0.525798,...,NaN,1,0.553851,0.553851,NaN,1,0.563766,0.563766,NaN,1
156,TFB,prices,DUET,1,0.089749,0.089585,0.001971,15,0.533924,0.533645,...,0.001625,15,0.544719,0.544107,0.001158,15,0.544283,0.543895,0.000994,15
38,MASTER,prices,MASTER,1,0.087109,0.087109,NaN,1,0.388889,0.388889,...,NaN,1,0.541302,0.541302,NaN,1,0.563843,0.563843,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
170,TFB,prices,Nonstationary_Transformer,10,0.240192,0.302394,0.105559,9,0.629098,0.655123,...,0.030262,9,0.616053,0.644065,0.051492,9,0.615176,0.643257,0.051407,9
176,TFB,prices,TimesNet,10,0.238484,0.276618,0.074792,9,0.642858,0.648543,...,0.019812,9,0.614913,0.634501,0.035396,9,0.613860,0.633470,0.035205,9
90,MASTER,return,MASTER,10,0.228867,0.228867,NaN,1,0.403571,0.403571,...,NaN,1,0.500996,0.500996,NaN,1,0.505715,0.505715,NaN,1
164,TFB,prices,FEDformer,10,0.228167,0.310717,0.132781,9,0.622391,0.650505,...,0.046322,9,0.610957,0.648539,0.067204,9,0.610343,0.647779,0.066912,9


In [9]:
# Comparação direta por janela de trading e modelo, usando medianas
if not metricas.empty:
    comp = (
        metricas
        .groupby(['janela_trading', 'grupo', 'modelo'], dropna=False)[[c for c in METRICAS_INTERESSE if c in metricas.columns]]
        .median()
        .reset_index()
        .sort_values(['janela_trading', 'mean_spearman_ic'], ascending=[True, False], na_position='last')
    )
    display(comp.head(100))


,janela_trading,grupo,modelo,mean_spearman_ic,mean_precision_positive,mean_precision_negative,auc_alta,auc_queda,auc_alta_media_janela,auc_queda_media_janela
2,1,TFB,DUET,0.062844,0.522379,NaN,0.525181,0.526554,0.531753,0.532277
7,1,TFB,TimesNet,0.047566,0.513305,NaN,0.519780,0.518704,0.524520,0.523943
6,1,TFB,Nonstationary_Transformer,0.027414,0.499025,NaN,0.513798,0.513645,0.515781,0.514362
4,1,TFB,FEDformer,0.021999,0.501287,NaN,0.507616,0.507824,0.510515,0.510246
1,1,MASTER,MASTER,0.008343,0.486842,NaN,0.507298,0.508505,0.508900,0.510522
0,1,Benchmark,k_1,-0.014542,0.488337,NaN,0.494015,0.493983,0.497892,0.499899
5,1,TFB,Momentum,-0.014542,0.485613,NaN,0.494015,0.493983,0.497892,0.499899
3,1,TFB,EqualWeight,NaN,0.491837,NaN,NaN,NaN,NaN,NaN
10,5,TFB,DUET,0.162078,0.589754,NaN,0.569883,0.570529,0.573436,0.574666
15,5,TFB,TimesNet,0.107125,0.567706,NaN,0.546051,0.545139,0.550333,0.549336


## Arquivos gerados

- `metricas_global_long.csv`: base longa, uma linha por JSON encontrado.
- `tabela_metricas_interesse.csv`: somente as métricas centrais.
- `resumo_por_modelo.csv`: média/mediana/desvio/contagem por grupo, dataset, modelo e janela de trading.
- `top_configs_por_metrica.csv`: melhores configurações por métrica.
- `ranking_global.csv`: ranking médio simples das métricas de interesse.
